Tahap 3 - Case Retrieval

Tujuan:

Mengubah representasi kasus menjadi vektor menggunakan TF-IDF

Membagi data menjadi data latih dan data uji (80:20)

Menghitung kemiripan antar kasus menggunakan Cosine Similarity

Menemukan Top-K kasus yang paling mirip dengan query kasus baru

Menyimpan query pengujian ke dalam queries.json

1. Import Library

In [27]:
import json
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModel
)

from sklearn.svm import LinearSVC

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.metrics.pairwise import (
    cosine_similarity
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

2. Load Dataset

In [28]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/processed/cases.csv"
)

print(df.shape)

df.head()

(50, 10)


,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full
0,1,case_001.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
1,2,case_002.txt,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30",gnomor 1092 k pid 2024 memeriksa perkara tinda...,dinyatakan ditolak dengan perbaikan e menimnba...,1230,8511,gnomor 1092 k pid 2024 memeriksa perkara tinda...
2,3,case_003.txt,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa...",gnomor 1153 pk pid.sus 2024 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,882,6185,gnomor 1153 pk pid.sus 2024 memeriksa perkara ...
3,4,case_004.txt,1171 pk pid.sus 2026,2026-01-05,"pasal 2, pasal 318, pasal 322, pasal 455, pasa...",gnomor 1171 pk pid.sus 2026 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,943,6493,gnomor 1171 pk pid.sus 2026 memeriksa perkara ...
4,5,case_005.txt,11809 k pid.sus 2025,2025-12-11,"pasal 2, pasal 253, pasal 296",gnomor 11809 k pid.sus 2025 memeriksa perkara ...,menolak permohonan kasasi dari pemohon kasasi ...,1354,9340,gnomor 11809 k pid.sus 2025 memeriksa perkara ...


3. Cek Missing Value

In [29]:
# ============================================================
# CEK MISSING VALUE
# ============================================================

df.isnull().sum()

case_id            0
file_name          0
no_perkara         0
tanggal            0
pasal              0
ringkasan_fakta    0
amar_putusan       0
jumlah_kata        0
jumlah_karakter    0
text_full          0
dtype: int64

4. Handle Missing Value

In [30]:
# ============================================================
# HANDLE MISSING VALUE
# ============================================================

df = df.fillna("")

print("Dataset bersih")

Dataset bersih


5. Perbaikan Amar Putusan

In [31]:
# ============================================================
# FIX OUTLIER AMAR PUTUSAN
# ============================================================

df.loc[
    df["case_id"] == 7,
    "amar_putusan"
] = (
    "permohonan peninjauan kembali dinyatakan ditolak "
    "dan putusan yang dimohonkan peninjauan kembali "
    "dinyatakan tetap berlaku"
)

df.loc[
    df["case_id"] == 19,
    "amar_putusan"
] = (
    "permohonan kasasi dinyatakan ditolak "
    "dengan perbaikan"
)

print(
    "Tidak ditemukan :",
    (df["amar_putusan"] == "Tidak Ditemukan").sum()
)

Tidak ditemukan : 0


6. Membuat Amar Singkat

In [32]:
# ============================================================
# AMAR PUTUSAN SINGKAT
# ============================================================

def simplify_amar(text):

    text = str(text).lower()

    if "ditolak dengan perbaikan" in text:
        return "ditolak dengan perbaikan"

    elif "dinyatakan ditolak" in text:
        return "dinyatakan ditolak"

    elif "menolak permohonan" in text:
        return "menolak permohonan"

    elif "mengabulkan permohonan" in text:
        return "mengabulkan permohonan"

    elif "mengadili" in text:
        return "mengadili"

    return "lainnya"


df["amar_singkat"] = df["amar_putusan"].apply(
    simplify_amar
)

df[
    [
        "no_perkara",
        "amar_singkat"
    ]
].head()

,no_perkara,amar_singkat
0,1013 k pid.sus 2026,mengadili
1,1092 k pid 2024,ditolak dengan perbaikan
2,1153 pk pid.sus 2024,dinyatakan ditolak
3,1171 pk pid.sus 2026,dinyatakan ditolak
4,11809 k pid.sus 2025,menolak permohonan


CELL 7 — Membuat Retrieval Text

Sesuai konsep CBR.

Gunakan:

Ringkasan Fakta + Pasal

karena itu identitas utama sebuah kasus.

In [33]:
# ============================================================
# RETRIEVAL TEXT
# ============================================================

df["retrieval_text"] = (

    df["pasal"].astype(str)

    + " "

    + df["ringkasan_fakta"].astype(str)

    + " "

    + df["text_full"].astype(str).str[:5000]

)

8. Train Test Split

In [34]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42

)

print("Train :", len(train_df))
print("Test  :", len(test_df))

Train : 40
Test  : 10


9. TF-IDF Vectorization

In [35]:
# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(

    max_features=5000,

    stop_words=None

)

X_train = vectorizer.fit_transform(

    train_df["retrieval_text"]

)

X_test = vectorizer.transform(

    test_df["retrieval_text"]

)

print("Shape Train :", X_train.shape)
print("Shape Test :", X_test.shape)

Shape Train : (40, 3213)
Shape Test : (10, 3213)


10. Similarity Matrix

In [36]:
# ============================================================
# SIMILARITY MATRIX
# ============================================================

similarity_matrix = cosine_similarity(

    X_train,

    X_train

)

print(similarity_matrix.shape)

(40, 40)


11. Fungsi Retrieval

In [37]:
# ============================================================
# RETRIEVE FUNCTION
# ============================================================

def retrieve(

    query,

    k=5

):

    query_vector = vectorizer.transform(

        [query]

    )

    similarity_scores = cosine_similarity(

        query_vector,

        X_train

    )[0]

    top_idx = np.argsort(

        similarity_scores

    )[::-1][:k]

    results = train_df.iloc[top_idx][

        [

            "case_id",

            "no_perkara",

            "tanggal",

            "pasal",

            "ringkasan_fakta",

            "amar_singkat"

        ]

    ].copy()

    results["similarity"] = (

        similarity_scores[top_idx]

    )

    return results

12. Query Test

In [38]:
# ============================================================
# QUERY TEST
# ============================================================

query_case = test_df.iloc[0]

print("NO PERKARA")
print(query_case["no_perkara"])

print("\nPASAL")
print(query_case["pasal"])

print("\nRINGKASAN FAKTA")
print(query_case["ringkasan_fakta"])

NO PERKARA
1684 k pid.sus 2026

PASAL
pasal   55, pasal 10, pasal 17, pasal 345, pasal 4, pasal 432, pasal 53, pasal 55

RINGKASAN FAKTA
gnomor 1684 k pid.sus 2026 memeriksa perkara tindak pidana khusus pada tingkat kasasi yang   dimohonkan oleh penuntut umum pada kejaksaan negeri sidoarjo, telah i memutus perkara terdakwa   nama mochamad baharudin amin a tempat lahir malang   m umur tanggal lahir 29 tahun 17b mei 1995 jenis kelamin laki lakui a kewarganegaraan indonesia   tempat tinggal jalan akordion, nomor 28, rt 002 rw 002,   a kelurahan tunggulwulung, kecamatan   i lowokwaru, kota malang   agama g islam e pekerjaan wiraswasta   terdakwa tersebut berada dalam tahanan rumah tahanan negara   rutan sejak tanggal 12 november 2024 sampai dengan sekarang   terdakwa diajukan di depan persidangan pengadilan negeri sidoarjo   karena didakwa dengan dakwaan sebagai berikut i kesatu perbuatan terdakwa sebagaimana diatur dan diancam pidana   dalam pasal 4 juncto pasal 10 undang undang nomor 21 

In [39]:
# ============================================================
# BUILD QUERY JSON
# ============================================================

queries = []

for idx in test_df.index[:10]:

    queries.append({

        "query_case_id":

        int(df.loc[idx,"case_id"]),

        "ground_truth":

        int(df.loc[idx,"case_id"])

    })

queries[:5]

[{'query_case_id': 14, 'ground_truth': 14},
 {'query_case_id': 40, 'ground_truth': 40},
 {'query_case_id': 31, 'ground_truth': 31},
 {'query_case_id': 46, 'ground_truth': 46},
 {'query_case_id': 18, 'ground_truth': 18}]

In [40]:
# ============================================================
# SAVE QUERY
# ============================================================

with open(

    "../data/eval/queries.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        queries,

        f,

        indent=4

    )

print("queries.json berhasil dibuat")

queries.json berhasil dibuat


13. Hasil Retrieval

In [41]:
# ============================================================
# TOP 5 SIMILAR CASES
# ============================================================

retrieve(

    query=query_case["retrieval_text"],

    k=5

)

,case_id,no_perkara,tanggal,pasal,ringkasan_fakta,amar_singkat,similarity
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",gnomor 1656 k pid.sus 2026 memeriksa perkara t...,ditolak dengan perbaikan,0.758485
12,13,1683 k pid.sus 2026,2026-02-11,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",gnomor 1683 k pid.sus 2026 memeriksa perkara t...,ditolak dengan perbaikan,0.737342
15,16,1930 k pid.sus 2026,2026-02-06,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81",fakta hukum yang relevan sebagai berikut bah...,ditolak dengan perbaikan,0.493519
16,17,1935 k pid.sus 2026,2025-10-28,"pasal 2, pasal 20, pasal 3, pasal 361, pasal 4...",nomor 1935 k pid.sus 2026 memeriksa perkara ti...,ditolak dengan perbaikan,0.487539
7,8,12069 k pid.sus 2025,2025-09-25,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 81",gnomor 12069 k pid.sus 2025 memeriksa perkara ...,mengadili,0.486997


### BAGIAN B

TF-IDF + SVM

In [42]:
# ============================================================
# EXPERIMENT 1
# TF-IDF + SVM
# ============================================================

from sklearn.svm import LinearSVC

In [43]:
# ============================================================
# TRAIN SVM
# ============================================================

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(

    X_train,

    train_df["amar_singkat"]

)

print(
    "SVM berhasil dilatih"
)

SVM berhasil dilatih


In [44]:
# ============================================================
# PREDIKSI SVM
# ============================================================

X_test = vectorizer.transform(

    test_df["retrieval_text"]

)

svm_pred = svm_model.predict(
    X_test
)

svm_result = pd.DataFrame({

    "case_id":
    test_df["case_id"].values,

    "actual":
    test_df["amar_singkat"].values,

    "predicted":
    svm_pred

})

svm_result.head()

,case_id,actual,predicted
0,14,ditolak dengan perbaikan,ditolak dengan perbaikan
1,40,dinyatakan ditolak,ditolak dengan perbaikan
2,31,mengadili,dinyatakan ditolak
3,46,dinyatakan ditolak,dinyatakan ditolak
4,18,ditolak dengan perbaikan,ditolak dengan perbaikan


In [45]:
# ============================================================
# EVALUASI SVM
# ============================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score

)

svm_accuracy = accuracy_score(

    test_df["amar_singkat"],

    svm_pred

)

svm_precision = precision_score(

    test_df["amar_singkat"],

    svm_pred,

    average="weighted",

    zero_division=0

)

svm_recall = recall_score(

    test_df["amar_singkat"],

    svm_pred,

    average="weighted",

    zero_division=0

)

svm_f1 = f1_score(

    test_df["amar_singkat"],

    svm_pred,

    average="weighted",

    zero_division=0

)

print(f"Accuracy  : {svm_accuracy:.2%}")
print(f"Precision : {svm_precision:.2%}")
print(f"Recall    : {svm_recall:.2%}")
print(f"F1 Score  : {svm_f1:.2%}")

Accuracy  : 50.00%
Precision : 30.00%
Recall    : 50.00%
F1 Score  : 37.14%


### BAGIAN C
IndoBERT Retrieval

In [46]:
# ============================================================
# LOAD INDOBERT
# ============================================================

MODEL_NAME = (
    "indobenchmark/indobert-base-p1"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModel.from_pretrained(
    MODEL_NAME
)

model.eval()

print(
    "IndoBERT berhasil dimuat"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33129.57it/s]

IndoBERT berhasil dimuat


In [47]:
# ============================================================
# EMBEDDING FUNCTION
# ============================================================

def get_embedding(text):

    text = str(text)

    inputs = tokenizer(

        text[:512],

        return_tensors="pt",

        truncation=True,

        padding=True,

        max_length=512

    )

    with torch.no_grad():

        outputs = model(**inputs)

    embedding = (

        outputs.last_hidden_state

        .mean(dim=1)

        .squeeze()

        .numpy()

    )

    return embedding

In [48]:
# ============================================================
# EMBEDDING TRAIN SET
# ============================================================

train_embeddings = np.array([

    get_embedding(text)

    for text in train_df["retrieval_text"]

])

print(
    train_embeddings.shape
)

(40, 768)


In [49]:
# ============================================================
# RETRIEVE BERT
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

def retrieve_bert(

    query,

    k=5

):

    query_embedding = get_embedding(
        query
    )

    similarities = cosine_similarity(

        [query_embedding],

        train_embeddings

    )[0]

    top_idx = np.argsort(
        similarities
    )[::-1][:k]

    results = train_df.iloc[
        top_idx
    ].copy()

    results["similarity"] = (
        similarities[top_idx]
    )

    return results

In [50]:
# ============================================================
# TEST RETRIEVAL BERT
# ============================================================

query_case = test_df.iloc[0]

retrieve_bert(

    query_case["retrieval_text"]

)[

    [
        "case_id",

        "no_perkara",

        "ringkasan_fakta",

        "similarity"

    ]

]

,case_id,no_perkara,ringkasan_fakta,similarity
11,12,1656 k pid.sus 2026,gnomor 1656 k pid.sus 2026 memeriksa perkara t...,0.961973
29,30,260 k pid.sus 2026,gnomor 260 k pid.sus 2026 memeriksa perkara ti...,0.954717
28,29,255 k pid.sus 2026,gnomor 255 k pid.sus 2026 memeriksa perkara ti...,0.953847
22,23,2052 k pid.sus 2026,gnomor 2052 k pid.sus 2026 memeriksa perkara t...,0.951701
12,13,1683 k pid.sus 2026,gnomor 1683 k pid.sus 2026 memeriksa perkara t...,0.951623


In [51]:
# ============================================================
# SUMMARY MODEL
# ============================================================

comparison_df = pd.DataFrame({

    "Model": [

        "TF-IDF + Cosine",

        "TF-IDF + SVM",

        "IndoBERT + Cosine"

    ],

    "Status": [

        "Ready",

        "Ready",

        "Ready"

    ]

})

comparison_df

,Model,Status
0,TF-IDF + Cosine,Ready
1,TF-IDF + SVM,Ready
2,IndoBERT + Cosine,Ready


In [52]:
# ============================================================
# SAVE TF-IDF VECTORIZER
# ============================================================

import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    vectorizer,
    "../models/tfidf_vectorizer.pkl"
)
joblib.dump(
    svm_model,
    "../models/svm_model.pkl"
)

print(
    "TF-IDF Vectorizer dan SVM Model berhasil disimpan"
)

TF-IDF Vectorizer dan SVM Model berhasil disimpan
